# KILN: threshold tuning from recordings

**Record:** `npm run dev`, open `http://127.0.0.1:5173/?dev=1&rec=1`. Keys `0–6` pick what you're doing
(`0 neutral · 1 shape · 2 pullUp · 3 pressDown · 4 point · 5 raise · 6 tooFast`), `R` starts/stops and downloads a JSON.
Put the files in `recordings/`. Record correct and deliberately sloppy versions, both of us, two lighting setups.

**Method:** same as picking a decision threshold on a validation set. For each feature, look at its distribution
for the class that should trigger the gesture vs the classes that shouldn't, and put the threshold in the gap.
Hysteresis: the ON threshold sits nearer the "yes" class, OFF nearer the "no" class, so values near the edge don't flicker.

In [ ]:
import glob
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FILES = sorted(glob.glob(str(ROOT / 'recordings' / '*.json')))

# current values in src/config.ts, drawn as dashed lines for comparison
CURRENT = {
    'PINCH_ON': 0.35, 'PINCH_OFF': 0.45,
    'FINGER_CURLED_ON': 0.35, 'FINGER_CURLED_OFF': 0.45,
    'FINGER_OPEN_ON': 0.60, 'FINGER_OPEN_OFF': 0.50,
    'MOTION_ON_PALM_PER_S': 0.40, 'MOTION_OFF_PALM_PER_S': 0.15, 'FULL_MOTION_PALM_PER_S': 1.5,
    'TEAR_SPEED_PALM_PER_S': 6.0, 'WOBBLE_TOL_PALM': 0.35, 'WOBBLE_CLEAR_TOL_PALM': 0.25,
}
FINGERS = ['index', 'middle', 'ring', 'pinky']

In [ ]:
def load(files):
    """One row per visible hand per observation."""
    rows = []
    for path in files:
        for fr in json.load(open(path, encoding='utf-8'))['frames']:
            inp, g = fr['input'], fr.get('gesture') or {}
            for side in ('screenLeft', 'screenRight'):
                h = inp.get(side)
                if not h:
                    continue
                ext = h['extension']
                v = h['velocityPalmPerS']
                rows.append({
                    'file': Path(path).name, 'label': fr['label'], 'side': side, 'tMs': inp['tMs'],
                    'status': inp['status'], 'pinch': h['pinchRatio'],
                    **{f'ext_{k}': ext[k] for k in FINGERS},
                    'ext_max': max(ext.values()), 'ext_min': min(ext.values()),
                    'pointing': h['pointing'],
                    'vy': v['y'] if h['velocityValid'] else np.nan,
                    'speed': float(np.hypot(v['x'], v['y'])) if h['velocityValid'] else np.nan,
                    'center': g.get('centerOffsetPalm'), 'recognized': g.get('gesture'),
                })
    return pd.DataFrame(rows)


def synthetic(n=300, seed=0):
    """Plausible fake data ONLY so the notebook runs end to end before real recordings exist."""
    rng = np.random.default_rng(seed)
    spec = {  # label: (pinch, ext_mean, vy, speed, center)
        'neutral': (0.8, 0.5, 0.0, 0.3, 0.1), 'shape': (0.9, 0.85, 0.0, 1.0, 0.15),
        'pullUp': (0.2, 0.45, 1.2, 1.3, 0.1), 'pressDown': (0.55, 0.15, -1.2, 1.3, 0.1),
        'raise': (0.9, 0.9, 0.0, 0.3, 0.1), 'tooFast': (0.9, 0.85, 0.0, 9.0, 0.2),
    }
    rows = []
    for label, (pinch, ext, vy, speed, center) in spec.items():
        for _ in range(n):
            e = np.clip(rng.normal(ext, 0.08, 4), 0, 1)
            rows.append({
                'file': 'SYNTHETIC', 'label': label, 'side': 'screenLeft', 'tMs': 0, 'status': 'ready',
                'pinch': max(0.0, rng.normal(pinch, 0.07)), **{f'ext_{k}': x for k, x in zip(FINGERS, e)},
                'ext_max': e.max(), 'ext_min': e.min(), 'pointing': False,
                'vy': rng.normal(vy, 0.3), 'speed': abs(rng.normal(speed, speed * 0.25)),
                'center': rng.normal(0, center), 'recognized': None,
            })
    return pd.DataFrame(rows)


df = load(FILES)
SYNTH = df.empty
if SYNTH:
    print('No recordings in recordings/: using SYNTHETIC data. Numbers below are NOT tuning results.')
    df = synthetic()
else:
    print(f'{len(FILES)} files, {len(df)} hand-frames')
df = df[df.status == 'ready'] if 'ready' in set(df.status) else df
print(df.groupby('label').size().to_string())

In [ ]:
def gap(yes, no, yes_below, p_yes=95, p_no=5):
    """ON/OFF thresholds in the gap between 'yes' and 'no' samples.
    yes_below=True: the gesture is ON when the value is LOW (e.g. pinch ratio when pinching).
    p_yes: % of 'yes' samples that must land on the ON side. 100 - p_no: % of 'no' samples on the OFF side."""
    yes, no = np.asarray(yes, float), np.asarray(no, float)
    yes, no = yes[~np.isnan(yes)], no[~np.isnan(no)]
    if len(yes) == 0 or len(no) == 0:
        return None
    if not yes_below:  # mirror so the same logic works for "ON when HIGH"
        r = gap(-yes, -no, True, p_yes, p_no)
        return r and {**r, 'on': -r['on'], 'off': -r['off'], 'edge_yes': -r['edge_yes'], 'edge_no': -r['edge_no']}
    edge_yes, edge_no = np.percentile(yes, p_yes), np.percentile(no, p_no)
    separable = edge_yes < edge_no
    lo, hi = (edge_yes, edge_no) if separable else sorted([np.median(yes), np.median(no)])
    return {'on': lo + (hi - lo) / 3, 'off': lo + 2 * (hi - lo) / 3,
            'edge_yes': edge_yes, 'edge_no': edge_no, 'separable': bool(separable)}


def plot(col, labels, lines, title):
    plt.figure(figsize=(8, 3))
    for label in labels:
        vals = df.loc[df.label == label, col].dropna()
        if len(vals):
            plt.hist(vals, bins=40, alpha=0.5, density=True, label=label)
    for name, (x, style) in lines.items():
        plt.axvline(x, linestyle=style, color='k', linewidth=1)
        plt.text(x, plt.ylim()[1] * 0.9, name, rotation=90, fontsize=7)
    plt.title(title + (' [SYNTHETIC]' if SYNTH else ''))
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.show()


at = lambda *labels: df.label.isin(labels)
suggest = {}

## Pinch (pullUp)
Also check `pressDown`: if a fist's thumb sits near the index tip, fists could read as pinch. Pinch is checked
BEFORE fist in the decision order, so that overlap would make pressDown impossible.

In [ ]:
r = gap(df.loc[at('pullUp'), 'pinch'], df.loc[at('shape', 'pressDown', 'raise', 'neutral'), 'pinch'], yes_below=True)
if r:
    suggest['PINCH_ON'], suggest['PINCH_OFF'] = r['on'], r['off']
    print(('separable' if r['separable'] else 'OVERLAP: check the plot'), r)
    fist_as_pinch = (df.loc[at('pressDown'), 'pinch'] < r['on']).mean()
    print(f'pressDown frames that would read as pinch: {fist_as_pinch:.1%}')
plot('pinch', ['pullUp', 'shape', 'pressDown', 'neutral'],
     {'current ON': (CURRENT['PINCH_ON'], '--'), **({'new ON': (r['on'], '-')} if r else {})}, 'pinch ratio')

## Fist (pressDown): the MOST extended finger must still be curled

In [ ]:
r = gap(df.loc[at('pressDown'), 'ext_max'], df.loc[at('shape', 'raise', 'point'), 'ext_max'], yes_below=True)
if r:
    suggest['FINGER_CURLED_ON'], suggest['FINGER_CURLED_OFF'] = r['on'], r['off']
    print(('separable' if r['separable'] else 'OVERLAP: check the plot'), r)
plot('ext_max', ['pressDown', 'shape', 'point', 'neutral'],
     {'current ON': (CURRENT['FINGER_CURLED_ON'], '--'), **({'new ON': (r['on'], '-')} if r else {})},
     'max finger extension')

## Open palm (shape / raise): the LEAST extended finger must still be open

In [ ]:
r = gap(df.loc[at('shape', 'raise'), 'ext_min'], df.loc[at('pressDown', 'pullUp'), 'ext_min'], yes_below=False)
if r:
    suggest['FINGER_OPEN_ON'], suggest['FINGER_OPEN_OFF'] = r['on'], r['off']
    print(('separable' if r['separable'] else 'OVERLAP: check the plot'), r)
plot('ext_min', ['shape', 'raise', 'pressDown', 'pullUp'],
     {'current ON': (CURRENT['FINGER_OPEN_ON'], '--'), **({'new ON': (r['on'], '-')} if r else {})},
     'min finger extension')

## Vertical motion (pull / press strength)

In [ ]:
moving = pd.concat([df.loc[at('pullUp'), 'vy'], -df.loc[at('pressDown'), 'vy']])
r = gap(moving, df.loc[at('shape', 'neutral'), 'vy'].abs(), yes_below=False, p_yes=90, p_no=10)
if r:
    suggest['MOTION_ON_PALM_PER_S'], suggest['MOTION_OFF_PALM_PER_S'] = r['on'], r['off']
    suggest['FULL_MOTION_PALM_PER_S'] = float(np.nanmedian(moving))
    print(r)
plot('vy', ['pullUp', 'pressDown', 'shape', 'neutral'], {'current ON': (CURRENT['MOTION_ON_PALM_PER_S'], '--')},
     'vertical velocity (palm/s, up = +)')

## Tear speed: calm shaping vs deliberately too fast
Err on the high side: a tear the user didn't cause is worse than a slightly late one.

In [ ]:
r = gap(df.loc[at('shape'), 'speed'], df.loc[at('tooFast'), 'speed'], yes_below=True, p_yes=99, p_no=25)
if r:
    suggest['TEAR_SPEED_PALM_PER_S'] = float(np.mean([r['edge_yes'], r['edge_no']]))
    print(r)
plot('speed', ['shape', 'tooFast'], {'current': (CURRENT['TEAR_SPEED_PALM_PER_S'], '--')}, 'hand speed (palm/s)')

## Wobble: how far off-centre calm, careful shaping already is (that must NOT trigger)

In [ ]:
c = df.loc[at('shape'), 'center'].dropna().abs()
if len(c):
    tol = float(np.percentile(c, 95)) * 1.2
    suggest['WOBBLE_TOL_PALM'], suggest['WOBBLE_CLEAR_TOL_PALM'] = tol, tol * 0.7
    print(f'p95 |centerOffset| while shaping: {np.percentile(c, 95):.2f} palm')
plot('center', ['shape', 'tooFast'], {'current': (CURRENT['WOBBLE_TOL_PALM'], '--')}, 'center offset (palm)')

## Labels vs what the recognizer said (real recordings only)

In [ ]:
if not SYNTH and df.recognized.notna().any():
    print(pd.crosstab(df.label, df.recognized, normalize='index').round(2).to_string())

## Suggested `config.ts` values
Paste only the ones whose plots show a real gap. Then use it again with your own hands before committing.

In [ ]:
print('// SYNTHETIC: do not paste' if SYNTH else f'// from {len(FILES)} recordings')
for k, v in suggest.items():
    print(f'  {k}: {v:.2f}, // was {CURRENT[k]}')